In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import sys, torch
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

import importlib, src.cbam
importlib.reload(src.cbam)

from src.cbam import CBAMVgg16, get_vgg_cbam_optimizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CBAMVgg16(num_classes=4, pretrained=True).to(device)
model.print_model_info()

dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    logits = model(dummy)

print(f"\nInput  : {dummy.shape}")
print(f"Output : {logits.shape}")  # expect [4, 4]
print("CBAMVgg16 ready for training.")

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

  CBAMVgg16 — Model Summary
  Backbone       : VGG16 (pretrained)
  CBAM params    : 33,410
  Total params   : 134,426,310
  Trainable now  : 165,766
  Dropout        : 0.3
  Output classes : 4

Input  : torch.Size([4, 3, 224, 224])
Output : torch.Size([4, 4])
CBAMVgg16 ready for training.


In [6]:
# ── Train CBAMVgg16
import os, torch, torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

from src.cbam import CBAMVgg16, get_vgg_cbam_optimizer
from src.dataset import create_dataloaders
from utils.logger import get_logger, MetricLogger

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'
EXP_DIR  = '/content/drive/MyDrive/brain_tumor_classification/experiments/cbam_vgg16'
os.makedirs(EXP_DIR, exist_ok=True)

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir=DATA_DIR, image_size=224, batch_size=32,
    val_split=0.2, seed=42,
)
print(f"Train: {info['train_size']} | Val: {info['val_size']}")

model     = CBAMVgg16(num_classes=4, dropout_rate=0.3, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

logger         = get_logger('cbam_vgg16', log_dir=EXP_DIR)
metric_tracker = MetricLogger(log_dir=EXP_DIR)

EPOCHS         = 30
WARMUP_EPOCHS  = 5
FINETUNE_EPOCH = 20
PATIENCE       = 7
best_val_f1    = 0.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):

    if epoch == 1:
        logger.info("PHASE 1 — backbone frozen, CBAM + head train")
        model.freeze_backbone()
        optimizer = get_vgg_cbam_optimizer(model, head_lr=1e-3, backbone_lr=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=WARMUP_EPOCHS, eta_min=1e-6)

    elif epoch == WARMUP_EPOCHS + 1:
        logger.info("PHASE 2 — top backbone blocks unfrozen")
        model.unfreeze_top_layers(num_blocks=2)
        optimizer = get_vgg_cbam_optimizer(model, head_lr=1e-4, backbone_lr=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=FINETUNE_EPOCH - WARMUP_EPOCHS, eta_min=1e-6)

    elif epoch == FINETUNE_EPOCH + 1:
        logger.info("PHASE 3 — full model unfrozen")
        model.unfreeze_all()
        optimizer = get_vgg_cbam_optimizer(model, head_lr=1e-5, backbone_lr=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(EPOCHS - FINETUNE_EPOCH, 1), eta_min=1e-6)

    # Training pass
    model.train()
    train_loss, train_preds, train_labels = 0.0, [], []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d} [Train]", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * images.size(0)
        train_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)
    train_acc   = accuracy_score(train_labels, train_preds)
    train_f1    = f1_score(train_labels, train_preds, average='macro', zero_division=0)

    # Validation pass
    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch:02d} [Val]  ", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits   = model(images)
            loss     = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            val_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_acc   = accuracy_score(val_labels, val_preds)
    val_f1    = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    scheduler.step()

    metric_tracker.update(epoch, 'train', train_loss, train_acc, train_f1)
    metric_tracker.update(epoch, 'val',   val_loss,   val_acc,   val_f1)

    logger.info(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1       = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(EXP_DIR, 'best_model.pth'))
        logger.info(f"  --> Best model saved (Val F1: {best_val_f1:.4f})")
    else:
        epochs_no_improve += 1
        logger.info(f"  No improvement. Patience: {epochs_no_improve}/{PATIENCE}")

    if epochs_no_improve >= PATIENCE:
        logger.info(f"Early stopping at epoch {epoch}.")
        break

metric_tracker.save()
logger.info(f"Training complete. Best Val F1: {best_val_f1:.4f}")
print(f"\nCBAM VGG16 training complete. Best Val F1: {best_val_f1:.4f}")

Train: 4480 | Val: 1120
2026-04-07 07:08:21 | INFO     | Log file: /content/drive/MyDrive/brain_tumor_classification/experiments/cbam_vgg16/cbam_vgg16_20260407_070821.log


INFO:cbam_vgg16:Log file: /content/drive/MyDrive/brain_tumor_classification/experiments/cbam_vgg16/cbam_vgg16_20260407_070821.log


2026-04-07 07:08:21 | INFO     | PHASE 1 — backbone frozen, CBAM + head train


INFO:cbam_vgg16:PHASE 1 — backbone frozen, CBAM + head train


Backbone frozen. CBAM + classifier training.
Trainable params: 165,766


...

2026-04-07 07:44:37 | INFO     |   No improvement. Patience: 1/7


INFO:cbam_vgg16:  No improvement. Patience: 1/7


2026-04-07 07:44:37 | INFO     | Training complete. Best Val F1: 0.9794


INFO:cbam_vgg16:Training complete. Best Val F1: 0.9794



CBAM VGG16 training complete. Best Val F1: 0.9794


In [7]:
# Evaluate CBAMVgg16 on test set and compare with VGG16 baseline
import yaml, os, torch, numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from src.cbam import CBAMVgg16
from src.dataset import BrainTumorDataset, CLASS_NAMES
from src.transforms import get_transforms
from src.evaluate import compute_metrics, plot_confusion_matrix, evaluate_experiment
import pandas as pd

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR    = '/content/drive/MyDrive/brain_tumor_classification/data'
EXP_DIR     = '/content/drive/MyDrive/brain_tumor_classification/experiments'
OUTPUTS_DIR = '/content/drive/MyDrive/brain_tumor_classification/outputs'

# ── Save config.yaml for cbam_vgg16
cbam_vgg_dir = os.path.join(EXP_DIR, 'cbam_vgg16')
config_dict  = {
    "name": "cbam_vgg16",
    "model": {"backbone": "vgg16", "pretrained": True, "num_classes": 4},
    "regularization": {"dropout_rate": 0.3, "use_mixup": False, "use_cutmix": False},
    "data": {"image_size": 224, "batch_size": 32, "val_split": 0.2,
             "seed": 42, "num_workers": 2},
    "train": {"epochs": 30, "warmup_epochs": 5, "warmup_lr": 1e-3,
              "finetune_lr": 1e-4, "full_finetune_lr": 1e-5,
              "full_finetune_epoch": 20, "optimizer": "adamw",
              "weight_decay": 1e-4, "scheduler": "cosine", "lr_min": 1e-6,
              "label_smoothing": 0.1, "patience": 7, "use_amp": True},
}
with open(os.path.join(cbam_vgg_dir, 'config.yaml'), 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False)

# ── Evaluate CBAMVgg16
print("Evaluating CBAMVgg16 on test set...")
model_cbam_vgg = CBAMVgg16(num_classes=4, dropout_rate=0.3, pretrained=False)
model_cbam_vgg.load_state_dict(torch.load(
    os.path.join(cbam_vgg_dir, 'best_model.pth'), map_location=device))
model_cbam_vgg = model_cbam_vgg.to(device)
model_cbam_vgg.eval()

test_loader = DataLoader(
    BrainTumorDataset(
        root_dir  = os.path.join(DATA_DIR, 'Testing'),
        transform = get_transforms(224, 'test'),
    ),
    batch_size=32, shuffle=False, num_workers=2
)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(device)
        logits  = model_cbam_vgg(images)
        probs   = torch.softmax(logits, dim=1)
        preds   = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

cbam_vgg_metrics = compute_metrics(all_preds, all_labels, np.array(all_probs))
plot_confusion_matrix(all_labels, all_preds,
                      experiment_name='cbam_vgg16',
                      outputs_dir=OUTPUTS_DIR)

print(f"\n  Accuracy  : {cbam_vgg_metrics['accuracy']:.2f}%")
print(f"  Macro F1  : {cbam_vgg_metrics['macro_f1']:.4f}")
print(f"  Macro AUC : {cbam_vgg_metrics['macro_auc']}")
print(f"\n  Per-class F1:")
for cls in CLASS_NAMES:
    print(f"    {cls:<15} {cbam_vgg_metrics[f'f1_{cls}']:.4f}")
print(f"\n{cbam_vgg_metrics['report']}")

# ── Load VGG16 baseline results
print("Loading VGG16 baseline results...")
vgg_results = evaluate_experiment(
    'baseline_vgg16', device=device, plot_cm=False)

# ── Final comparison table
rows = [
    {"Model": "VGG16 (baseline)",
     "Accuracy (%)": vgg_results["accuracy"],
     "Macro F1":     vgg_results["macro_f1"],
     "Macro AUC":    vgg_results["macro_auc"],
     "F1 Glioma":    vgg_results["f1_glioma"],
     "F1 Meningioma":vgg_results["f1_meningioma"],
     "F1 Notumor":   vgg_results["f1_notumor"],
     "F1 Pituitary": vgg_results["f1_pituitary"]},
    {"Model": "VGG16 + CBAM (proposed)",
     "Accuracy (%)": cbam_vgg_metrics["accuracy"],
     "Macro F1":     cbam_vgg_metrics["macro_f1"],
     "Macro AUC":    cbam_vgg_metrics["macro_auc"],
     "F1 Glioma":    cbam_vgg_metrics["f1_glioma"],
     "F1 Meningioma":cbam_vgg_metrics["f1_meningioma"],
     "F1 Notumor":   cbam_vgg_metrics["f1_notumor"],
     "F1 Pituitary": cbam_vgg_metrics["f1_pituitary"]},
]
df = pd.DataFrame(rows)

print("\n" + "=" * 80)
print("  FINAL COMPARISON — VGG16 vs VGG16 + CBAM (TEST SET)")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

print("\n  Per-class F1 change:")
for cls in ["glioma", "meningioma", "notumor", "pituitary"]:
    diff = cbam_vgg_metrics[f"f1_{cls}"] - vgg_results[f"f1_{cls}"]
    print(f"    {cls:<15} {diff:+.4f}")
print(f"\n  Macro F1 change : "
      f"{cbam_vgg_metrics['macro_f1'] - vgg_results['macro_f1']:+.4f}")
print(f"  Accuracy change : "
      f"{cbam_vgg_metrics['accuracy'] - vgg_results['accuracy']:+.2f}%")

Evaluating CBAMVgg16 on test set...

  Accuracy  : 94.00%
  Macro F1  : 0.9390
  Macro AUC : 0.9866

  Per-class F1:
    glioma          0.8883
    meningioma      0.9187
    notumor         0.9614
    pituitary       0.9875

              precision    recall  f1-score   support

      glioma       0.98      0.81      0.89       400
  meningioma       0.88      0.96      0.92       400
     notumor       0.93      1.00      0.96       400
   pituitary       0.99      0.99      0.99       400

    accuracy                           0.94      1600
   macro avg       0.94      0.94      0.94      1600
weighted avg       0.94      0.94      0.94      1600

Loading VGG16 baseline results...

  Evaluating: baseline_vgg16
  Backbone  : vgg16
  Device    : cuda
  Test set  : 1600 images

  Accuracy  : 94.31%
  Macro F1  : 0.9421
  Macro AUC : 0.9925

  Per-class F1:
    glioma          0.8919
    meningioma      0.9205
    notumor         0.9721
    pituitary       0.9838

  Classification Rep

In [6]:
!pip install grad-cam

In [12]:
# Load VGG16 baseline and CBAMVgg16 for Grad-CAM comparison
import torch, os, numpy as np, cv2, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

from src.gradcam import load_model_from_experiment, collect_samples, denormalize_image
from src.cbam import CBAMVgg16
from src.dataset import BrainTumorDataset, CLASS_NAMES
from src.transforms import get_transforms

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR    = '/content/drive/MyDrive/brain_tumor_classification/data'
EXP_DIR     = '/content/drive/MyDrive/brain_tumor_classification/experiments'
OUTPUTS_DIR = '/content/drive/MyDrive/brain_tumor_classification/outputs/gradcam'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Load VGG16 baseline
print("Loading VGG16 baseline...")
model_vgg, _ = load_model_from_experiment('baseline_vgg16', device)
layer_vgg    = model_vgg.backbone.features[-1]
print("  Done.")

# Load CBAMVgg16
print("Loading CBAMVgg16...")
model_cbam_vgg = CBAMVgg16(num_classes=4, dropout_rate=0.3, pretrained=False)
model_cbam_vgg.load_state_dict(torch.load(
    os.path.join(EXP_DIR, 'cbam_vgg16', 'best_model.pth'),
    map_location=device))
model_cbam_vgg = model_cbam_vgg.to(device)
model_cbam_vgg.eval()
layer_cbam_vgg = model_cbam_vgg.backbone.features[-1]
print("  Done.")

# Build test loader
test_loader = DataLoader(
    BrainTumorDataset(
        root_dir  = os.path.join(DATA_DIR, 'Testing'),
        transform = get_transforms(224, 'test'),
    ),
    batch_size=16, shuffle=True, num_workers=2
)

# Collect 4 misclassified samples from VGG16 baseline
print("\nCollecting misclassified samples from VGG16 baseline...")
_, incorrect_vgg = collect_samples(
    model_vgg, test_loader, device,
    n_correct=0, n_incorrect=4
)
print(f"  Collected {len(incorrect_vgg)} samples.")

Loading VGG16 baseline...
  Done.
Loading CBAMVgg16...
  Done.

  Collected 4 samples.


In [13]:
def generate_gradcam_fixed(model, target_layer, image_tensor, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    input_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        logits     = model(input_tensor)
        probs      = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        pred_prob  = probs[0, pred_class].item()

    # Temporarily enable gradients for all params
    original_requires_grad = {}
    for name, param in model.named_parameters():
        original_requires_grad[name] = param.requires_grad
        param.requires_grad_(True)
    try:
        cam = GradCAM(model=model, target_layers=[target_layer])
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)
        grayscale_cam = grayscale_cam[0]
    finally:
        for name, param in model.named_parameters():
            param.requires_grad_(original_requires_grad[name])

    original_image = denormalize_image(image_tensor)
    cam_image = show_cam_on_image(
        original_image, grayscale_cam,
        use_rgb=True, colormap=cv2.COLORMAP_JET, image_weight=0.5)
    cam_image = cam_image.astype(np.float32) / 255.0
    return cam_image, pred_class, pred_prob, grayscale_cam


def plot_comparison_fixed(model_a, layer_a, model_b, layer_b,
                          samples, label_a, label_b, device,
                          save_path=None):
    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.5))
    if n == 1:
        axes = axes[None, :]

    fig.suptitle(f"Grad-CAM: {label_a}  vs  {label_b}",
                 fontsize=13, fontweight='bold', y=1.01)

    col_titles = ["Original MRI",
                  f"{label_a}\nGrad-CAM",
                  f"{label_b}\nGrad-CAM"]
    for col, ct in enumerate(col_titles):
        axes[0, col].set_title(ct, fontsize=10, fontweight='bold', pad=10)

    for row, (img_tensor, true_label, _) in enumerate(samples):
        true_name = CLASS_NAMES[true_label].capitalize()

        original = denormalize_image(img_tensor)
        axes[row, 0].imshow(original, cmap='gray')
        axes[row, 0].set_ylabel(f"True: {true_name}", fontsize=9,
                                fontweight='bold', rotation=0,
                                labelpad=70, va='center')
        axes[row, 0].axis('off')

        cam_a, pred_a, prob_a, _ = generate_gradcam_fixed(
            model_a, layer_a, img_tensor, device)
        color_a = '#2ca02c' if pred_a == true_label else '#d62728'
        axes[row, 1].imshow(cam_a)
        axes[row, 1].set_title(
            f"Pred: {CLASS_NAMES[pred_a].capitalize()} ({prob_a:.0%})",
            fontsize=9, color=color_a)
        axes[row, 1].axis('off')

        cam_b, pred_b, prob_b, _ = generate_gradcam_fixed(
            model_b, layer_b, img_tensor, device)
        color_b = '#2ca02c' if pred_b == true_label else '#d62728'
        axes[row, 2].imshow(cam_b)
        axes[row, 2].set_title(
            f"Pred: {CLASS_NAMES[pred_b].capitalize()} ({prob_b:.0%})",
            fontsize=9, color=color_b)
        axes[row, 2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close()
        print(f"Saved → {save_path}")
    else:
        plt.show()

print("Functions ready.")

Functions ready.


In [15]:
# Generate VGG16 vs CBAMVgg16 Grad-CAM comparison
plot_comparison_fixed(
    model_a   = model_vgg,
    layer_a   = layer_vgg,
    model_b   = model_cbam_vgg,
    layer_b   = layer_cbam_vgg,
    samples   = incorrect_vgg,
    label_a   = "VGG16 (baseline)",
    label_b   = "VGG16 + CBAM",
    device    = device,
    save_path = os.path.join(OUTPUTS_DIR, 'gradcam_vgg16_vs_cbam_vgg16.png'),
)

Saved → /content/drive/MyDrive/brain_tumor_classification/outputs/gradcam/gradcam_vgg16_vs_cbam_vgg16.png
